# 28/ Loan Default Prediction

**Dataset (Kaggle):** [https://www.kaggle.com/datasets/nikhil1e9/loan-default](https://www.kaggle.com/datasets/nikhil1e9/loan-default)

> Loan application records with borrower financial/demographic features and a Default status (target).

**Goal:** Predict whether a borrower will default on a loan.

**How to get the data:**
1. Download from the link above (or run `kaggle datasets download -d <dataset-slug> -p data/ --unzip`, requires a `kaggle.json` API token in `~/.kaggle/`).
2. Place the extracted file(s) in a local `data/` folder next to this notebook.
3. Update `DATA_PATH` in the cell below if your filename differs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from sklearn.ensemble import RandomForestClassifier

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)


In [ ]:
DATA_PATH = "data/Loan_Default.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


## Exploratory Data Analysis

In [ ]:
df.info()
df.describe(include='all').T


In [ ]:
print(df['Status'].value_counts(normalize=True))
sns.countplot(x='Status', data=df)
plt.title("Target distribution")
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
num_df = df.select_dtypes(include=[np.number])
sns.heatmap(num_df.corr(), cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.show()


## Preprocessing

Check the exact target column name after loading — some versions of this dataset call it 'Status' or 'Default'; adjust the `target` variable in the preprocessing cell if needed.

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=["LoanID"], errors='ignore')
df_clean = df_clean.dropna()

# Encode target if it's categorical text
if df_clean['Status'].dtype == object:
    le = LabelEncoder()
    df_clean['Status'] = le.fit_transform(df_clean['Status'])

# One-hot encode remaining categorical features
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
df_clean = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

X = df_clean.drop(columns=['Status'])
y = df_clean['Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(X_train.shape, X_test.shape)


## Model Training

In [ ]:
model = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=42, class_weight="balanced")
model.fit(X_train_scaled, y_train)


## Evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1] if len(set(y_test)) == 2 else None

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

if y_proba is not None:
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.show()


## Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(15).plot(kind='barh', figsize=(8,6))
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importances")
plt.show()


## Next Steps
- Tune hyperparameters with `GridSearchCV` / `Optuna`.
- Try `XGBoost` / `LightGBM` and compare.
- Handle class imbalance with `SMOTE` if needed.
- Save the model with `joblib.dump(model, 'model.pkl')` for deployment.